<a href="https://colab.research.google.com/github/GabyPugaBR/agentic-ai-for-developers-concepts-and-applications-for-enterprises-3913172/blob/main/AgenticAI_LlamaIndex.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install python-dotenv==1.0.0
!pip install llama-index==0.10.59
!pip install llama-index-llms-openai==0.1.27
!pip install llama-index-embeddings-openai==0.1.11

  Using cached python_dotenv-1.0.0-py3-none-any.whl.metadata (21 kB)
Using cached python_dotenv-1.0.0-py3-none-any.whl (19 kB)
  Attempting uninstall: python-dotenv
    Found existing installation: python-dotenv 1.2.1
    Uninstalling python-dotenv-1.2.1:
      Successfully uninstalled python-dotenv-1.2.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
llama-cloud-services 0.6.54 requires python-dotenv<2,>=1.0.1, but you have python-dotenv 1.0.0 which is incompatible.
google-adk 1.25.1 requires tenacity<10.0.0,>=9.0.0, but you have tenacity 8.5.0 which is incompatible.
  Using cached llama_index-0.10.59-py3-none-any.whl.metadata (11 kB)
  Using cached llama_index_cli-0.1.13-py3-none-any.whl.metadata (1.5 kB)
  Using cached llama_index_core-0.10.59-py3-none-any.whl.metadata (2.4 kB)
  Using cached llama_index_embeddings_openai-0.1.11-py3-none-any.whl.metadata (6

In [2]:
!pip install llama-index --upgrade
!pip install llama-index-llms-openai --upgrade
!pip install llama-index-embeddings-openai --upgrade

  Using cached llama_index-0.14.15-py3-none-any.whl.metadata (13 kB)
  Using cached llama_index_cli-0.5.3-py3-none-any.whl.metadata (1.4 kB)
  Using cached llama_index_core-0.14.15-py3-none-any.whl.metadata (2.6 kB)
  Using cached llama_index_embeddings_openai-0.5.1-py3-none-any.whl.metadata (400 bytes)
  Using cached llama_index_indices_managed_llama_cloud-0.9.4-py3-none-any.whl.metadata (3.7 kB)
  Using cached llama_index_llms_openai-0.6.21-py3-none-any.whl.metadata (3.0 kB)
  Using cached llama_index_readers_file-0.5.6-py3-none-any.whl.metadata (5.7 kB)
  Using cached llama_index_readers_llama_parse-0.5.1-py3-none-any.whl.metadata (3.1 kB)
  Using cached pypdf-6.7.4-py3-none-any.whl.metadata (7.1 kB)
  Using cached llama_parse-0.6.94-py3-none-any.whl.metadata (7.1 kB)
  Using cached llama_cloud_services-0.6.94-py3-none-any.whl.metadata (3.8 kB)
INFO: pip is looking at multiple versions of llama-cloud-services to determine which version is compatible with other requirements. This cou

In [3]:

from llama_index.llms.openai import OpenAI
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.core import Settings
from google.colab import userdata

Settings.llm = OpenAI(
    model="gpt-4o-mini",  # ← correct model string
    api_key=userdata.get('OpenAI_API')
)

Settings.embed_model = OpenAIEmbedding(
    model="text-embedding-3-small",
    api_key=userdata.get('OpenAI_API')
)

In [4]:
#Create indexes for vector search
from llama_index.core import SimpleDirectoryReader
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core import  VectorStoreIndex

splitter=SentenceSplitter(chunk_size=1024)

#-------------------------------------------------------------------
#Setup Aeroflow document index
#-------------------------------------------------------------------
aeroflow_documents=SimpleDirectoryReader(
    input_files=["AeroFlow_Specification_Document.pdf"])\
            .load_data()

#Read documents into nodes
aeroflow_nodes=splitter.get_nodes_from_documents(aeroflow_documents)
#Create a vector Store
aeroflow_index=VectorStoreIndex(aeroflow_nodes)
#Create a query engine
aeroflow_query_engine = aeroflow_index.as_query_engine()

#-------------------------------------------------------------------
#Setup EchoSprint document index
#-------------------------------------------------------------------
ecosprint_documents=SimpleDirectoryReader(
    input_files=["EcoSprint_Specification_Document.pdf"])\
            .load_data()
#Read documents into nodes
ecosprint_nodes=splitter.get_nodes_from_documents(ecosprint_documents)
#Create a vector Store
ecosprint_index=VectorStoreIndex(ecosprint_nodes)
#Create a query engine
ecosprint_query_engine = ecosprint_index.as_query_engine()

In [7]:
from llama_index.core.tools import QueryEngineTool
from llama_index.core.query_engine.router_query_engine import RouterQueryEngine
from llama_index.core.selectors import PydanticSingleSelector  # ← changed

# Create a query engine Tool for AeroFlow
aeroflow_tool = QueryEngineTool.from_defaults(
    query_engine=aeroflow_query_engine,
    name="Aeroflow specifications",
    description=(
        "Contains information about Aeroflow: Design, features, technology, maintenance, warranty"
    ),
)

# Create a query engine Tool for EcoSprint
ecosprint_tool = QueryEngineTool.from_defaults(
    query_engine=ecosprint_query_engine,
    name="EcoSprint specifications",
    description=(
        "Contains information about EcoSprint: Design, features, technology, maintenance, warranty"
    ),
)

# Create a Router Agent
router_agent = RouterQueryEngine(
    selector=PydanticSingleSelector.from_defaults(),  # ← changed
    query_engine_tools=[
        aeroflow_tool,
        ecosprint_tool,
    ],
    verbose=True
)

In [8]:
#Ask a question about NoSQL
response = router_agent.query("What colors are available for AeroFlow?")
print("\nResponse: ",str(response))

Selecting query engine 0: The question pertains to Aeroflow, and the first choice contains information about Aeroflow, which may include details about its design and features, potentially covering available colors..

Response:  The AeroFlow is available in Coastal Blue, Sunset Orange, and Pearl White.


In [9]:
response = router_agent.query("What colors are available for EcoSprint?")
print("\nResponse: ",str(response))

Selecting query engine 1: The question specifically asks about EcoSprint, and choice 2 contains information about EcoSprint, which may include details about available colors..

Response:  The EcoSprint is available in Midnight Black, Ocean Blue, and Pearl White.
